In [654]:
import pandas as pd
import numpy as np

In [655]:
df_sq = pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\squads.csv')
df_bts=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\batting_stats.csv')
df_bs=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\bowling_stats.csv')
df_pt=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\points_table.csv')
df_mt=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\matches.csv')
 #added prev years data from 2008-onwards

print(df_sq.isnull().sum())
print(df_bts.isnull().sum())
print(df_bs.isnull().sum())
print(df_pt.isnull().sum())
print(df_mt.isnull().sum())

#drop the row in which match was abandoned 
df_mt=df_mt[df_mt['match_result']=='completed']
print(df_mt.isnull().sum())
df_mt = df_mt[df_mt['match_result'] == 'completed']
df_mt['date'] = pd.to_datetime(df_mt['date'], format='%B %d, %Y')   

team_no          0
team_name        0
player           0
nationality      0
role             0
designation    247
dtype: int64
position       0
batsman        0
team           0
matches        0
innings        0
runs           0
impact         0
average        0
strike_rate    0
not_outs       0
high_score     0
balls_faced    0
hundreds       0
fifties        0
ducks          0
fours          0
sixes          0
dtype: int64
position        0
bowler          0
team            0
matches         0
innings         0
wickets         0
economy         0
impact          0
bbi             0
overs           0
balls           0
dot_balls       0
maiden_overs    0
runs            0
avg             0
4wh             0
5wh             0
dtype: int64
position     0
team         0
matches      0
wins         0
defeats      0
ties         0
abandoned    0
points       0
nrr          0
dtype: int64
match_id                0
date                    0
venue                   0
team1                   0


In [656]:


df_bs['dotball_ratio'] = df_bs['dot_balls'] / df_bs['balls']
#dropping all the features not needed by us for our model
df_sq=df_sq[['team_name','player']]
df_bts=df_bts[['batsman','average','impact','strike_rate']]
df_bs=df_bs[['bowler','avg','impact','economy','dotball_ratio']]
df_mt=df_mt[['date','team1','team2','match_winner']]
df_pt=df_pt[['team','points','nrr']]


#features

#headtohead, teamrating, batting strength , bowling stregth, previous match performances

df_pred=pd.DataFrame(columns=['Team Name','Team Rating','Batting Strength','Bowling Strength','Star Presence'])
#made a new df and added the 
df_pred['Team Name']=df_pt['team']
print(df_pred['Team Name'])


#headtohead of two teams dataframe 


0                   Punjab Kings
1    Royal Challengers Bengaluru
2            Sunrisers Hyderabad
3               Rajasthan Royals
4                 Gujarat Titans
5            Chennai Super Kings
6                 Delhi Capitals
7          Kolkata Knight Riders
8                 Mumbai Indians
9           Lucknow Super Giants
Name: Team Name, dtype: object


In [657]:
#BAtting strength of a team and bowling strength of a team

df_sq_batting = df_sq.merge(df_bts, left_on='player', right_on='batsman', how='left')
df_sq_bowling = df_sq.merge(df_bs, left_on='player', right_on='bowler', how='left')

team_batting_strength = df_sq_batting.groupby('team_name')[['average', 'strike_rate']].mean()
team_batting_strength['Batting Strength'] = (
    team_batting_strength['average'] * 0.5 + team_batting_strength['strike_rate'] * 0.5
)

team_bowling_strength = df_sq_bowling.groupby('team_name')[['avg', 'economy']].mean()
team_bowling_strength['Bowling Strength'] = (
    (1 / team_bowling_strength['avg']) * 0.5 + (1 / team_bowling_strength['economy']) * 0.5
)

df_pred['Batting Strength'] = df_pred['Team Name'].map(team_batting_strength['Batting Strength'])
df_pred['Bowling Strength'] = df_pred['Team Name'].map(team_bowling_strength['Bowling Strength'])

print(df_pred)
print("\nMissing values:\n", df_pred.isnull().sum())

                     Team Name Team Rating  Batting Strength  \
0                 Punjab Kings         NaN          132.8500   
1  Royal Challengers Bengaluru         NaN          110.5000   
2          Sunrisers Hyderabad         NaN          117.3200   
3             Rajasthan Royals         NaN          118.8975   
4               Gujarat Titans         NaN               NaN   
5          Chennai Super Kings         NaN          110.2450   
6               Delhi Capitals         NaN               NaN   
7        Kolkata Knight Riders         NaN               NaN   
8               Mumbai Indians         NaN               NaN   
9         Lucknow Super Giants         NaN               NaN   

   Bowling Strength Star Presence  
0               NaN           NaN  
1          0.085544           NaN  
2          0.080424           NaN  
3          0.082735           NaN  
4               NaN           NaN  
5          0.081363           NaN  
6               NaN           NaN  
7      

In [658]:
#check if a top performer from either bowling rating or batting rating is present in the squad od one team 

#is the union of the star performers of bnatting and bowling of the entire tournament
star_players=set(df_bts['batsman']).union(set(df_bs['bowler']))


df_sq['is_star']=df_sq['player'].isin(star_players)

df_pred['Star Presence']=df_pred['Team Name'].map(df_sq.groupby('team_name')['is_star'].sum())


df_pred['Star Presence']=df_pred['Star Presence'].fillna(0)

#completed my star presence feature and now moving on to team rating and squad strength features

team_rating = (df_pt['points']*0.2 + df_pt['nrr']).set_axis(df_pt['team'])
df_pred['Team Rating']=df_pred['Team Name'].map(team_rating)
print(df_pred)

#completed my team rating feature and now moving on to squad strength features

                     Team Name  Team Rating  Batting Strength  \
0                 Punjab Kings        3.933          132.8500   
1  Royal Challengers Bengaluru        4.319          110.5000   
2          Sunrisers Hyderabad        2.815          117.3200   
3             Rajasthan Royals        2.602          118.8975   
4               Gujarat Titans        1.125               NaN   
5          Chennai Super Kings        1.079          110.2450   
6               Delhi Capitals        0.140               NaN   
7        Kolkata Knight Riders        1.751               NaN   
8               Mumbai Indians        0.064               NaN   
9         Lucknow Super Giants       -0.306               NaN   

   Bowling Strength  Star Presence  
0               NaN            3.0  
1          0.085544            4.0  
2          0.080424            4.0  
3          0.082735            4.0  
4               NaN            0.0  
5          0.081363            3.0  
6               NaN      

In [659]:
# Load the extra historical matches
df_mt_prev = pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\matches_prev_years.csv')
df_mt_prev['date'] = pd.to_datetime(df_mt_prev['date'], format='%Y-%m-%d')   # ADD THIS LINE, right after loading df_mt_prev
# Extract a clean 4-digit year from season, handling both "2015" and "2007/08" formats
df_mt_prev['season_year'] = df_mt_prev['season'].str.extract(r'(\d{4})').astype(int)

# Keep only the last 7 seasons
df_mt_prev = df_mt_prev[df_mt_prev['season_year'] >= 2018]

# Map old/retired franchise names to their current names
team_rename_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}
df_mt_prev['team1'] = df_mt_prev['team1'].replace(team_rename_map)
df_mt_prev['team2'] = df_mt_prev['team2'].replace(team_rename_map)
df_mt_prev['winner'] = df_mt_prev['winner'].replace(team_rename_map)

# Align columns to match df_mt's structure, then combine
df_mt_prev = df_mt_prev[['date', 'team1', 'team2', 'winner']].rename(columns={'winner': 'match_winner'})
df_mt_combined = pd.concat(
    [df_mt_prev, df_mt[['date', 'team1', 'team2', 'match_winner']]],
    ignore_index=True
)
df_mt_combined['date'] = pd.to_datetime(df_mt_combined['date'])
df_mt_combined = df_mt_combined.sort_values('date').reset_index(drop=True)

print(df_mt_combined.shape)

print(df_mt_combined['season_year'].min() if 'season_year' in df_mt_combined else "combined ok")
df_headtohead=pd.DataFrame(columns=['Team1','Team2',' Winner'])


(497, 4)
combined ok


In [667]:
# ============================================================
# Head-to-head feature + match features, using FULL team names 
# consistently across current season and historical data
# ============================================================

team_name_map = {
    'RCB': 'Royal Challengers Bengaluru', 'SRH': 'Sunrisers Hyderabad',
    'MI': 'Mumbai Indians', 'KKR': 'Kolkata Knight Riders',
    'PBKS': 'Punjab Kings', 'RR': 'Rajasthan Royals',
    'GT': 'Gujarat Titans', 'LSG': 'Lucknow Super Giants',
    'DC': 'Delhi Capitals', 'CSK': 'Chennai Super Kings'
}

# Step 1: Map this season's abbreviations to full names
df_mt['team1_full'] = df_mt['team1'].map(team_name_map)
df_mt['team2_full'] = df_mt['team2'].map(team_name_map)
df_mt['match_winner_full'] = df_mt['match_winner'].map(team_name_map)

print("Unmapped team1/team2 check (should all be 0):")
print(df_mt[['team1_full', 'team2_full']].isnull().sum())

# Step 2: Merge team1's and team2's pre-computed features (rating, star presence etc.)
df_matches_features = df_mt.merge(
    df_pred, left_on='team1_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team1_rating', 'Batting Strength': 'team1_batting_strength',
    'Bowling Strength': 'team1_bowling_strength', 'Star Presence': 'team1_star_presence'
}).drop(columns=['Team Name'])

df_matches_features = df_matches_features.merge(
    df_pred, left_on='team2_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team2_rating', 'Batting Strength': 'team2_batting_strength',
    'Bowling Strength': 'team2_bowling_strength', 'Star Presence': 'team2_star_presence'
}).drop(columns=['Team Name'])

# Step 3: Build the target variable
df_matches_features['team1_won'] = (
    df_matches_features['match_winner'] == df_matches_features['team1']
).astype(int)

df_matches_features = df_matches_features.sort_values('date').reset_index(drop=True)



# Step 4: Build a FULL combined history (historical + current season), all using full team names
df_mt_current_full = df_mt[['date', 'team1_full', 'team2_full', 'match_winner_full']].rename(
    columns={'team1_full': 'team1', 'team2_full': 'team2', 'match_winner_full': 'match_winner'}
)

df_mt_combined = pd.concat([df_mt_prev, df_mt_current_full], ignore_index=True)
df_mt_combined = df_mt_combined.sort_values('date').reset_index(drop=True)

print("\nCombined history shape:", df_mt_combined.shape)

# Step 5: Head-to-head lookup, filtered by DATE (not row index), using full names
def get_head_to_head(team1, team2, match_date, history_df):
    h2h_matches = history_df[
        (history_df['date'] < match_date) &
        (((history_df['team1'] == team1) & (history_df['team2'] == team2)) |
         ((history_df['team1'] == team2) & (history_df['team2'] == team1)))
    ]
    total = len(h2h_matches)
    if total == 0:
        return 0.5
    team1_wins = (h2h_matches['match_winner'] == team1).sum()
    return team1_wins / total

df_matches_features['team1_h2h_win_rate'] = df_matches_features.apply(
    lambda row: get_head_to_head(row['team1_full'], row['team2_full'], row['date'], df_mt_combined),
    axis=1
)

# Step 6: NOW it's safe to drop the helper columns, since h2h calc is done
df_matches_features = df_matches_features.drop(columns=['team1_full', 'team2_full', 'match_winner_full'])

print(df_matches_features.head())
print("\n", df_matches_features[['date', 'team1', 'team2', 'match_winner', 'team1_h2h_win_rate']].head(15))
print("\nh2h value distribution:\n", df_matches_features['team1_h2h_win_rate'].value_counts())

Unmapped team1/team2 check (should all be 0):
team1_full    0
team2_full    0
dtype: int64

Combined history shape: (497, 4)
        date team1 team2 match_winner  team1_rating  team1_batting_strength  \
0 2026-03-28   RCB   SRH          RCB         4.319                110.5000   
1 2026-03-29    MI   KKR           MI         0.064                     NaN   
2 2026-03-30    RR   CSK           RR         2.602                118.8975   
3 2026-03-31  PBKS    GT         PBKS         3.933                132.8500   
4 2026-04-01   LSG    DC           DC        -0.306                     NaN   

   team1_bowling_strength  team1_star_presence  team2_rating  \
0                0.085544                  4.0         2.815   
1                     NaN                  0.0         1.751   
2                0.082735                  4.0         1.079   
3                     NaN                  3.0         1.125   
4                0.102700                  2.0         0.140   

   team2_battin

In [663]:
df_current_season = df_mt.merge(
    df_pred, left_on='team1_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team1_rating', 'Batting Strength': 'team1_batting_strength',
    'Bowling Strength': 'team1_bowling_strength', 'Star Presence': 'team1_star_presence'
}).drop(columns=['Team Name'])

df_current_season = df_current_season.merge(
    df_pred, left_on='team2_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team2_rating', 'Batting Strength': 'team2_batting_strength',
    'Bowling Strength': 'team2_bowling_strength', 'Star Presence': 'team2_star_presence'
}).drop(columns=['Team Name'])

df_current_season['team1_won'] = (
    df_current_season['match_winner'] == df_current_season['team1']
).astype(int)

df_current_season = df_current_season.sort_values('date').reset_index(drop=True)

print("Current season shape:", df_current_season.shape)
print(df_current_season.isnull().sum())

Current season shape: (38, 16)
date                       0
team1                      0
team2                      0
match_winner               0
team1_full                 0
team2_full                 0
match_winner_full          0
team1_rating               0
team1_batting_strength    19
team1_bowling_strength    18
team1_star_presence        0
team2_rating               0
team2_batting_strength    19
team2_bowling_strength    18
team2_star_presence        0
team1_won                  0
dtype: int64


In [664]:
#grt recent rolling form of the team 
def get_recent_form(team, match_date, history_df, n=10):
    team_matches = history_df[
        ((history_df['team1'] == team) | (history_df['team2'] == team)) &
        (history_df['date'] < match_date)
    ].tail(n)
    if len(team_matches) == 0:
        return 0.5
    wins = (team_matches['match_winner'] == team).sum()
    return wins / len(team_matches)



df_current_season['team1_h2h_win_rate'] = df_current_season.apply(
    lambda row: get_head_to_head(row['team1_full'], row['team2_full'], row['date'], df_mt_combined),
    axis=1
)

df_current_season['team1_form'] = df_current_season.apply(
    lambda row: get_recent_form(row['team1_full'], row['date'], df_mt_combined), axis=1
)
df_current_season['team2_form'] = df_current_season.apply(
    lambda row: get_recent_form(row['team2_full'], row['date'], df_mt_combined), axis=1
)

print(df_current_season[['date', 'team1', 'team2', 'team1_h2h_win_rate', 'team1_form', 'team2_form']])

         date team1 team2  team1_h2h_win_rate  team1_form  team2_form
0  2026-03-28   RCB   SRH            0.500000         0.6         0.5
1  2026-03-29    MI   KKR            0.538462         0.3         0.8
2  2026-03-30    RR   CSK            0.583333         0.5         0.5
3  2026-03-31  PBKS    GT            0.400000         0.3         0.4
4  2026-04-01   LSG    DC            0.600000         0.4         0.6
5  2026-04-02   KKR   SRH            0.687500         0.7         0.4
6  2026-04-03   CSK  PBKS            0.461538         0.4         0.4
7  2026-04-04    GT    RR            0.833333         0.3         0.5
8  2026-04-04    DC    MI            0.466667         0.7         0.3
9  2026-04-05   SRH   LSG            0.250000         0.5         0.4
10 2026-04-05   RCB   CSK            0.307692         0.7         0.3
11 2026-04-07    RR    MI            0.615385         0.5         0.3
12 2026-04-08    DC    GT            0.600000         0.7         0.3
13 2026-04-09   KKR 

In [665]:
feature_columns_combined = [
    'team1_rating', 'team2_rating',
    #'team1_batting_strength', 'team2_batting_strength',
    #'team1_bowling_strength', 'team2_bowling_strength',
    'team1_star_presence', 'team2_star_presence',
    'team1_h2h_win_rate',
    'team1_form', 'team2_form'
]

X = df_current_season[feature_columns_combined]
y = df_current_season['team1_won']

print("Feature matrix shape:", X.shape)
print("Missing values:\n", X.isnull().sum())

Feature matrix shape: (38, 7)
Missing values:
 team1_rating           0
team2_rating           0
team1_star_presence    0
team2_star_presence    0
team1_h2h_win_rate     0
team1_form             0
team2_form             0
dtype: int64


In [666]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

split_index = int(len(df_current_season) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

importance_df = pd.DataFrame({
    'feature': feature_columns_combined,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)
print("\nFeature importance:\n", importance_df)

results_df = X_test.copy()
results_df['actual_team1_won'] = y_test.values
results_df['predicted_team1_won'] = y_pred
results_df['team1_win_probability'] = y_pred_proba[:, 1]
results_df['team2_win_probability'] = y_pred_proba[:, 0]

print("\nTest set predictions with probabilities:\n",
      results_df[['actual_team1_won', 'predicted_team1_won', 'team1_win_probability', 'team2_win_probability']])


Train size: 30, Test size: 8

Accuracy: 0.625

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.80      0.73         5
           1       0.50      0.33      0.40         3

    accuracy                           0.62         8
   macro avg       0.58      0.57      0.56         8
weighted avg       0.60      0.62      0.60         8


Confusion Matrix:
 [[4 1]
 [2 1]]

Feature importance:
                feature  coefficient
0         team1_rating     0.658549
6           team2_form     0.258089
1         team2_rating     0.231551
2  team1_star_presence     0.230501
4   team1_h2h_win_rate     0.118702
5           team1_form    -0.231399
3  team2_star_presence    -0.316251

Test set predictions with probabilities:
     actual_team1_won  predicted_team1_won  team1_win_probability  \
30                 1                    0               0.193950   
31                 0                    0               0.159155   
32       